# 01 · Images, datasets, and trustworthy splits

**Level:** beginner · **Prerequisite:** lesson 00 · **Teaching time:** 40–60 minutes.

Build the data pipeline before building a model. You will inspect real clothing images,
understand pixel/channel/batch dimensions, distinguish training from validation and test data,
and calculate normalization statistics from training examples only.

Dataset: **Fashion-MNIST**, 28×28 grayscale images in ten clothing classes.
Its original training and test partitions have 60,000 and 10,000 examples.
First run downloads roughly 30 MB; cached runs work offline.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## Step 1 — A color image is three grids

Start with a generated RGB image so every channel has an obvious meaning. Pillow and plotting
usually use **HWC**; PyTorch vision models use **CHW**, plus a batch axis for multiple examples.


In [ ]:
from PIL import Image
from torchvision import transforms

rgb = np.zeros((32, 48, 3), dtype=np.uint8)
rgb[:, :16, 0] = 255
rgb[:, 16:32, 1] = 255
rgb[:, 32:, 2] = 255
rgb_image = Image.fromarray(rgb)
tensor_image = transforms.ToTensor()(rgb_image)
print("NumPy HWC:", rgb.shape, rgb.dtype)
print("PyTorch CHW:", tensor_image.shape, tensor_image.dtype)
print("Pixel range:", tensor_image.min().item(), tensor_image.max().item())
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(rgb_image)
axes[0].set_title("RGB combined")
for channel, name, ax in zip(tensor_image, ["Red", "Green", "Blue"], axes[1:]):
    ax.imshow(channel, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name + " channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 2 — Set the workload explicitly

We use small, reproducible, class-balanced subsets for fast teaching. The subset is useful for
learning but is not the complete benchmark. Download size stays the same even when training
uses a subset: torchvision first caches the original dataset.


In [ ]:
TRAIN_LIMIT = 120 if SMOKE else (3000 if QUICK else None)
VAL_LIMIT = 60 if SMOKE else (800 if QUICK else None)
TEST_LIMIT = 60 if SMOKE else (1000 if QUICK else None)
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_limit=TRAIN_LIMIT, val_limit=VAL_LIMIT, test_limit=TEST_LIMIT)
print(dict(train=len(train_ds), validation=len(val_ds), test=len(test_ds)))
print(dict(enumerate(class_names)))


## Step 3 — Inspect images and labels together

Always look at the data before training. Ask whether labels are plausible, the images are readable,
and the task is possible from the available pixels. In this dataset a shirt and a T-shirt may look
similar at 28×28 resolution.

**Demonstrate:** ask the audience to classify three images before revealing their labels.


In [ ]:
cu.show_images(train_ds, class_names, n=12)
image, label = train_ds[0]
print("Image shape:", tuple(image.shape), "label ID:", label, "name:", class_names[label])
assert image.shape == (1, 28, 28)
assert image.dtype == torch.float32
assert 0 <= image.min() <= image.max() <= 1


## Step 4 — Zoom in until an image becomes numbers

An 8×8 crop below is a small matrix of brightness values. The model receives numbers, not a file name
or a human description. `ToTensor()` scales ordinary uint8 image values from `[0, 255]` to `[0, 1]`.


In [ ]:
crop = image[0, 10:18, 10:18]
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(crop, cmap="gray", vmin=0, vmax=1)
for row in range(8):
    for col in range(8):
        ax.text(col, row, f"{crop[row, col]:.1f}", ha="center", va="center",
                color="black" if crop[row, col] > 0.5 else "white", fontsize=8)
ax.set_title("An 8×8 crop: brightness as numbers")
plt.show()


## Step 5 — Separate learning, choosing, and reporting

| Split | Allowed use |
|---|---|
| Training | Fit model weights and data statistics |
| Validation | Choose hyperparameters, model, and stopping epoch |
| Test | Report performance after those choices are fixed |

Our helper splits only the original training partition into training and validation, stratified by
class. A stratified split keeps class coverage. The official test partition is separate.
Different transforms wrap separate dataset objects, so changing training augmentation cannot
accidentally augment validation images.

For real data, split by **person, product, recording session, or time** when images share a source.
Random image-level splitting can leak near-duplicate or same-subject information.


In [ ]:
train_indices = set(train_ds.indices)
val_indices = set(val_ds.indices)
assert train_indices.isdisjoint(val_indices)
assert train_ds.dataset.train and val_ds.dataset.train
assert not test_ds.dataset.train
print("Train/validation source indices are disjoint.")
print("Test comes from the official held-out partition.")
# Index 5 in the test partition is NOT the same source image as index 5 in training.


## Step 6 — Check class balance and a no-learning baseline

Accuracy is the fraction of correct predictions. A model that always predicts the most common
class establishes a useful baseline. On strongly imbalanced data it can look impressive while
never recognizing a minority class. We inspect training and validation counts here, leaving test
performance for later modeling lessons.


In [ ]:
def subset_labels(subset):
    return np.asarray(subset.dataset.targets)[np.asarray(subset.indices)]

train_labels = subset_labels(train_ds)
val_labels = subset_labels(val_ds)
train_counts = np.bincount(train_labels, minlength=len(class_names))
val_counts = np.bincount(val_labels, minlength=len(class_names))
majority_class = int(train_counts.argmax())
print("Majority-class validation accuracy:", np.mean(val_labels == majority_class))
x = np.arange(len(class_names))
plt.figure(figsize=(11, 4))
plt.bar(x - .2, train_counts, .4, label="Train")
plt.bar(x + .2, val_counts, .4, label="Validation")
plt.xticks(x, class_names, rotation=45, ha="right")
plt.ylabel("Example count")
plt.legend()
plt.tight_layout()
plt.show()


## Step 7 — Turn single examples into batches

A `Dataset` retrieves one `(image, label)` pair. A `DataLoader` groups those pairs into batches,
optionally shuffling the order each epoch. Training is shuffled; evaluation usually is not.
We set `num_workers=0` to keep the examples reliable inside Windows notebooks.


In [ ]:
train_loader = cu.loader(train_ds, batch_size=32, shuffle=True)
images, labels = next(iter(train_loader))
print("Images NCHW:", tuple(images.shape))
print("Labels N:", tuple(labels.shape), labels.dtype)
print("Batches per epoch:", len(train_loader))
print("First label names:", [class_names[i] for i in labels[:6].tolist()])
assert images.shape[0] == labels.shape[0]
assert labels.dtype == torch.long


## Step 8 — Calculate normalization from training data only

Scaling to `[0, 1]` and standardizing to `(pixel - mean) / std` are different operations.
Estimate mean and standard deviation from training pixels, then reuse the same values on
validation, test, and future input images. Here we use a bounded sample for demonstration.
For a pretrained model, follow its published preprocessing instead (lesson 06).


In [ ]:
sample_loader = cu.loader(train_ds, batch_size=128, shuffle=False)
pixel_sum = torch.zeros(1, dtype=torch.float64)
pixel_square_sum = torch.zeros(1, dtype=torch.float64)
pixel_count = 0
for batch_index, (x, _) in enumerate(sample_loader):
    x = x.double()
    pixel_sum += x.sum(dim=(0, 2, 3))
    pixel_square_sum += x.square().sum(dim=(0, 2, 3))
    pixel_count += x.shape[0] * x.shape[2] * x.shape[3]
    if batch_index == 9:  # at most 1,280 training images
        break
mean = pixel_sum / pixel_count
std = (pixel_square_sum / pixel_count - mean.square()).clamp_min(1e-12).sqrt()
print("Training-sample mean:", mean.tolist(), "std:", std.tolist())
normalizer = transforms.Normalize(mean.tolist(), std.tolist())
normalized = normalizer(image)
reconstructed = normalized * std.float().view(-1, 1, 1) + mean.float().view(-1, 1, 1)
assert torch.allclose(image, reconstructed, atol=1e-6)
print("Normalized range:", normalized.min().item(), normalized.max().item())


## Step 9 — See how augmentation changes a training example

Augmentation simulates plausible changes without changing the label. It belongs in the training
pipeline. Validation and test preprocessing should be deterministic so comparisons are meaningful.
An augmentation that destroys the true label is harmful; ask what transformations make sense for
your own domain before choosing them.


In [ ]:
augment = transforms.RandomAffine(degrees=12, translate=(.08, .08))
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i, ax in enumerate(axes):
    candidate = image if i == 0 else augment(image)
    ax.imshow(candidate[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title("Original" if i == 0 else f"Variation {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 10 — Inspect the helper instead of treating it as magic

The course keeps repeated plumbing in `course_utils.py`. Early model lessons write the training
loop explicitly. Read the split implementation below and identify where the random seed is used.


In [ ]:
import inspect
print(inspect.getsource(cu.split_indices))


## Exercises and live questions

1. Change batch size from 32 to 7. What shape does the last batch have?
2. Replace the augmentation with a vertical flip. Is the result plausible for clothing photos?
3. Your dataset has ten photos of each person. How should you split it?
4. Why does normalization use training statistics rather than all available images?

<details><summary>Suggested answers</summary>

1. Inspect `list(cu.loader(train_ds, batch_size=7))[-1][0].shape`; the final batch may contain fewer than seven examples.
2. Usually not representative of the expected input; augmentation depends on the task.
3. Keep every photo of one person in one partition, using a group-aware split.
4. Otherwise the training pipeline uses information from held-out data and no longer represents a clean evaluation.

</details>

**Ready to move on:** describe one image, one batch, and the separate jobs of all three splits.

**Primary references:** [Fashion-MNIST project](https://github.com/zalandoresearch/fashion-mnist),
[datasets and dataloaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html),
[torchvision transforms](https://docs.pytorch.org/vision/stable/transforms.html).
